# Registration of Two-Photon Autofluorescence Images and H&E staining images

## Step 1: Pre-process TPAF image and H&E image

TPAF: Convert to grayscale, invert color, rescaling to match H&E scale

H&E: Convert to grayscale

In [ ]:
# General
import os

import tifffile as tiff
import cv2
import numpy as np 
from skimage.feature import match_template
import matplotlib.pyplot as plt
import json
 
# module for improving low-light region signal
from skimage import exposure, img_as_float
from scipy.ndimage import gaussian_filter

# Step 2
from skimage import feature, transform, measure, color, registration
from skimage.feature import ORB, match_descriptors, plot_matches
from skimage.transform import warp, AffineTransform
import SimpleITK as sitk



#### TPAF 预处理


#### Low-Light Signal Enhancement Functions

In [ ]:
def flat_field_correct(raw, flat=None, dark=None):
    """
    原始图像 raw，flat-field 图，dark 图均为 numpy 数组
    返回校正后的图像（float）
    """
    R = raw.astype(np.float32)
    if dark is not None:
        R = R - dark.astype(np.float32)
    if flat is not None:
        F = flat.astype(np.float32)
        if dark is not None:
            F = F - dark.astype(np.float32)
        m = np.mean(F)
        # 防止除零
        gain = m / (F + 1e-6)
        corrected = R * gain
    else:
        corrected = R
    # clip 到非负
    corrected = np.clip(corrected, 0, None)
    # 归一化到 [0,1]
    corrected = corrected / np.max(corrected)
    return corrected

def retrospective_background_subtract(img, sigma=50):
    """
    无校正时用 Gaussian 滤波估背景后减法
    """
    background = gaussian_filter(img, sigma=sigma)
    res = img - background
    res += np.mean(background)  # 加回均值以保亮度动态
    res = np.clip(res, 0, None)
    res = res / np.max(res)
    return res

def apply_clahe(img, clip_limit=0.01, kernel_size=None):
    """
    CLAHE 局部对比度增强
    img: float 图像，介于 [0,1]
    返回增强后的 float 图像
    """
    img_uint8 = np.uint8(img * 255)
    clahe = cv2.createCLAHE(clipLimit=clip_limit*255, tileGridSize=(kernel_size or (8,8)))
    cl = clahe.apply(img_uint8)
    return cl.astype(np.float32) / 255.0

def adjust_gamma(img, gamma=0.5):
    """Gamma 校正，gamma <1 增亮暗区"""
    return exposure.adjust_gamma(img, gamma=gamma)

def enhance_fluorescence(raw, flat=None, dark=None,
                         gauss_sigma=50,
                         clahe_clip=0.01, clahe_grid=(8,8),
                         gamma=0.8):
    # 1. Flat‑field 校正 / 背景剔除
    if flat is not None:
        # rescale flat 和 dark 到 raw 的大小
        if flat.shape != raw.shape:
            flat = cv2.resize(flat, (raw.shape[1], raw.shape[0]), interpolation=cv2.INTER_CUBIC)
        if dark.shape != raw.shape:
            dark = cv2.resize(dark, (raw.shape[1], raw.shape[0]), interpolation=cv2.INTER_CUBIC)
        img = flat_field_correct(raw, flat=flat, dark=dark)
    else:
        img = raw.astype(np.float32)
        img = img / np.max(img)
        img = retrospective_background_subtract(img, sigma=gauss_sigma)
    # 2. CLAHE 增强
    img = apply_clahe(img, clip_limit=clahe_clip, kernel_size=clahe_grid)
    # 3. Gamma 增亮弱信号
    img = adjust_gamma(img, gamma=gamma)
    return img

ref_im_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
# load flat field and dark reference images
flat_field_im_name = "flat_field_ref_im.tif"
flat_field_im = tiff.imread(os.path.join(ref_im_dir, flat_field_im_name))
dark_im_name = "dark_ref_im.tif"
dark_im = tiff.imread(os.path.join(ref_im_dir, dark_im_name))
#flat_field_im = None
#dark_field_im = None


1. 尺度变换为与 H&E 图像一致
2. 转换为 grayscale
3. 弱光区域的信号增强
4. 反转颜色

In [ ]:
root_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
tpaf_im_dir = os.path.join(root_dir, "00_og_TPAF")
output_tpaf_dir = os.path.join(root_dir, "01_TPAF_rescaled_gray")
if not os.path.exists(output_tpaf_dir):
    os.makedirs(output_tpaf_dir)

tpaf_im_list = sorted([f for f in os.listdir(tpaf_im_dir) if f.endswith('.tif')])

# process TPAF images
for i in range(len(tpaf_im_list)):
    tpaf_im = tiff.imread(os.path.join(tpaf_im_dir, tpaf_im_list[i]))

    # rescaling TPAF image (structures in TPAF images are 0.39x smaller than in H&E, need to rescale to same level as H&E)
    rescale_factor = 1 / 0.39
    tpaf_im_rescaled = cv2.resize(tpaf_im, None, fx=rescale_factor, fy=rescale_factor, interpolation=cv2.INTER_CUBIC)

    # convert to grayscale
    tpaf_im_gray = cv2.cvtColor(tpaf_im_rescaled, cv2.COLOR_RGB2GRAY)
    #tiff.imwrite(os.path.join(output_tpaf_dir, tpaf_im_list[i]), tpaf_im_rescaled_inverted)

    # Enhance fluorescence
    enhanced = enhance_fluorescence(tpaf_im_gray, flat=flat_field_im, dark=dark_im,
                                    gauss_sigma=100,
                                    clahe_clip=0.01,
                                    clahe_grid=(16,16),
                                    gamma=0.7)
    # convert to uint8
    enhanced = (enhanced * 255).astype(np.uint8)

    # invert color
    tpaf_im_inverted = 255 - enhanced  # invert color
    
    tiff.imwrite(os.path.join(output_tpaf_dir, tpaf_im_list[i]), tpaf_im_inverted)
    print(f"Processed TPAF image: {tpaf_im_list[i]}")

#### H&E 预处理
1. 转换为 grayscale

In [ ]:
root_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
he_im_dir = os.path.join(root_dir, "00_og_HE")
output_he_dir = os.path.join(root_dir, "01_HE_inverted")
if not os.path.exists(output_he_dir):
    os.makedirs(output_he_dir)

he_im_list = sorted([f for f in os.listdir(he_im_dir) if f.endswith('.tif')])

# process H&E images
for i in range(len(he_im_list)):
    he_im = tiff.imread(os.path.join(he_im_dir, he_im_list[i]))
    # convert to grayscale
    he_im_gray = cv2.cvtColor(he_im, cv2.COLOR_RGB2GRAY)
    tiff.imwrite(os.path.join(output_he_dir, he_im_list[i]), he_im_gray)
    print(f"Processed H&E image: {he_im_list[i]}")

## Step 2: Correlation Patch Mathing

Find, crop and save the same FOV in H&E image as TPAF image.

In [ ]:
# Correlation patch matching

root_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
TPAF_FOV_name = "20240829 750nm 20mW 240817HOC240827-4 Line-02_0002"
source_tpaf_dir = os.path.join(root_dir, "01_TPAF_rescaled_gray")
target_he_dir = os.path.join(root_dir, "01_HE_gray")
output_dir = os.path.join(root_dir, "02_correlation_matching_results/" + TPAF_FOV_name)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

source_tpaf_name = TPAF_FOV_name + ".tif"
target_he_name = "HE_line_01_02.tif"
source_tpaf_path = os.path.join(source_tpaf_dir, source_tpaf_name)
target_he_path = os.path.join(target_he_dir, target_he_name)

# set a redundancy factor for the increasing patch size to include more content at four borders
# this is to ensure that the patch is large enough to cover as many features in the TPAF ROI as possible
redundancy_factor = 0.03

def find_best_patch(tpaf_gray, he_gray, patch_size=1024):
    """
    tpaf_gray: template patch, shape (patch_size, patch_size)
    he_gray: whole-slide H&E grayscale image, shape (H, W)
    returns: best_match_coords (y, x), max_score, result_matrix
    """
    result = match_template(he_gray, tpaf_gray, pad_input=False)  # normalized cross‑correlation
    ij = np.unravel_index(np.argmax(result), result.shape)
    y, x = ij
    max_score = result[y, x]
    return (y, x), max_score, result

print("Loading TPAF and H&E images for correlation matching...")
print(f"Source TPAF path: {source_tpaf_path}")
print(f"Target H&E path: {target_he_path}")
TPAF_im = tiff.imread(source_tpaf_path)
HE_im = tiff.imread(target_he_path)

# enhance TPAF image contrast
TPAF_im = cv2.normalize(TPAF_im, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

patch_size = TPAF_im.shape[0]  # assuming square patches
redundancy_size = int(patch_size * redundancy_factor)
assert TPAF_im.shape[0] == TPAF_im.shape[1], "TPAF image must be square"

coords, score, result = find_best_patch(TPAF_im, HE_im, patch_size=patch_size)
y, x = coords[0] - redundancy_size, coords[1] - redundancy_size
y = max(0, y)  # ensure y is not negative
x = max(0, x)  # ensure x is not negative
print(f"Best match at (row, col) = {coords}, NCC score = {score:.4f}")

# crop matched region from H&E
patch_size_with_redundancy = patch_size + 2 * redundancy_size
if y + patch_size_with_redundancy > HE_im.shape[0]:
    y = HE_im.shape[0] - patch_size_with_redundancy
if x + patch_size_with_redundancy > HE_im.shape[1]:
    x = HE_im.shape[1] - patch_size_with_redundancy
print(f"Cropping H&E patch at (row, col) = ({y}, {x}), size = {patch_size_with_redundancy}")
he_patch = HE_im[y:y+patch_size_with_redundancy, x:x+patch_size_with_redundancy]
tiff.imwrite(os.path.join(output_dir, source_tpaf_name[:-4]+"_matched_HE_patch.tif"), he_patch)

# save y, x, score, patch size with redundancy to a json file named after the TPAF image
output_txt = os.path.join(output_dir, source_tpaf_name[:-4]+"_matching_info.json")
matching_info = {
    "y": y,
    "x": x,
    "score": score,
    "patch_size_with_redundancy": patch_size_with_redundancy
}
with open(output_txt, 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    matching_info_serializable = {k: int(v) if isinstance(v, (np.integer, np.int64, np.int32)) else float(v) if isinstance(v, (np.floating, np.float64, np.float32)) else v for k, v in matching_info.items()}
    json.dump(matching_info_serializable, f, indent=4)
# visualize the results
print(f"Results saved to {output_txt}")

fig, ax = plt.subplots(1, 3, figsize=(15,5))
ax[0].imshow(TPAF_im, cmap='gray')
ax[0].set_title('TPAF Template')
ax[1].imshow(he_patch, cmap='gray')
ax[1].set_title('Matched H&E Patch')
ax[2].imshow(HE_im, cmap='gray')
ax[2].add_patch(plt.Rectangle((x,y), patch_size, patch_size,
                edgecolor='r', facecolor='none', linewidth=2))
ax[2].set_title('H&E Whole-slide with matched box')
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()


## Step 2: Global Registration

旋转校正（Global Registration）

由于样本在两次成像（荧光与明场）中的放置可能存在微小误差（如1–2°角度偏差），仍需进行进一步的全局旋转校正：

从图像对中提取特征描述子（feature descriptors）及其位置。

利用描述子匹配相似特征点。

使用 M-estimator sample consensus (MSAC) 算法（一种 RANSAC 变体）拟合匹配点对，估计变换矩阵。（Hartley, R. & Zisserman, A. Multiple View Geometry in Computer Vision (Cambridge Univ. Press, Cambridge, 2003).）

应用该矩阵将原始明场图像进行旋转校正。

对校正后的图像边缘再裁剪100个像素（每边裁50），以避免旋转后产生的空白像素。


#### 方法 1：ORB + RANSAC

In [ ]:
def global_register(source_TAPF_img, target_HE_img, og_TPAF_img,
                    n_keypoints=500, residual_thresh=3, max_trials=1000):
    """
    source_TAPF_img: grayscale or inverted TPAF image
    target_HE_img: grayscale H&E image (matched scale)
    Returns: corrected_TAPF (rotated), transform_model, inlier_mask
    """
    # Initiate feature extractor
    print("Feature point extraction...")
    # ORB feature extraction
    # n_keypoints: number of keypoints to detect
    orb = ORB(n_keypoints=n_keypoints, fast_threshold=0.05) # ORB feature extractor
    orb.detect_and_extract(source_TAPF_img)
    kp_TPAF = orb.keypoints
    desc_TPAF = orb.descriptors
    orb.detect_and_extract(target_HE_img)
    kp_HE = orb.keypoints
    desc_HE = orb.descriptors

    # descriptor matching
    print("Descriptor matching...")
    # cross_check=True: only keep matches where both descriptors match each other
    matches = match_descriptors(desc_HE, desc_TPAF, cross_check=True)
    src = kp_HE[matches[:,0]]
    dst = kp_TPAF[matches[:,1]]
    
    print("Transformation estimation with RANSAC...")
    # Estimate EuclideanTransform with RANSAC（rigid: rotation + translation）
    model_robust, inliers = measure.ransac(
        (src, dst),
        transform.EuclideanTransform,
        min_samples=3,
        residual_threshold=residual_thresh,
        max_trials=max_trials,
        is_model_valid=None,
        is_data_valid=None
    )
    
    print("Total number of keypoints in TPAF:", len(kp_TPAF))
    print("Total number of keypoints in HE:", len(kp_HE))
    print("inliers:", inliers)
    print(f"Number of matches: {len(matches)}, Number of inliers: {np.sum(inliers)}")
    print(f"Estimated rotation (rad): {model_robust.rotation}, translation: {model_robust.translation}")

    # 使用 warp 应用 transform 的 inverse（将 HE 图像校正）
    print("Applying transformation to warp/correct TPAF image...")
    corrected = transform.warp(
        og_TPAF_img,
        model_robust.inverse,
        order=1, mode='constant', cval=0, preserve_range=True
    )

    # 裁剪每边 50 像素以去除 rotation 空白
    h, w = corrected.shape[:2]
    #cropped = corrected[50:h-50, 50:w-50]
    cropped = corrected

    return cropped, model_robust, inliers, src, dst

if __name__ == '__main__':
    
    root_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
    TPAF_FOV_name = "20240829 750nm 20mW 240817HOC240827-4 Line-01_0002"
    og_TPAF_dir = os.path.join(root_dir, "00_og_TPAF")
    source_tpaf_dir = os.path.join(root_dir, "01_TPAF_rescaled_gray")
    target_he_dir = os.path.join(root_dir, "02_correlation_matching_results/" + TPAF_FOV_name)
    output_dir = os.path.join(root_dir, "03_global_registration_results/" + TPAF_FOV_name)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    source_tpaf_name = TPAF_FOV_name + ".tif"
    target_he_name = source_tpaf_name[:-4] + "_matched_HE_patch.tif"
    og_tpaf_path = os.path.join(og_TPAF_dir, source_tpaf_name)
    source_tpaf_path = os.path.join(source_tpaf_dir, source_tpaf_name)
    target_he_path = os.path.join(target_he_dir, target_he_name)

    print("Loading TPAF, H&E images for global registration...")
    og_tpaf_im = tiff.imread(og_tpaf_path)
    source_tpaf_im = tiff.imread(source_tpaf_path)
    target_he_im = tiff.imread(target_he_path)

    corr, model, inliers, src, dst = global_register(source_tpaf_im, target_he_im, og_tpaf_im, n_keypoints=800, residual_thresh=5)

    # Saving corrected TPAF image
    corrected_tpaf_path = os.path.join(output_dir, TPAF_FOV_name + "_corrected.tif")
    tiff.imwrite(corrected_tpaf_path, corr.astype(np.uint8))

    print("Estimated rotation (rad):", model.rotation, "translation:", model.translation)
    plt.figure(figsize=(10,6))
    plt.subplot(121)
    plt.title('Matched keypoints (inliers shown)')
    plot_matches(
        plt.gca(),
        source_tpaf_im, target_he_im,
        src[inliers], dst[inliers],
        np.column_stack((np.arange(np.sum(inliers)), np.arange(np.sum(inliers)))),
        only_matches=True
    )
    plt.axis('off')
    plt.subplot(122)
    plt.title('Registered TPAF (cropped)')
    plt.imshow(corr, cmap='gray')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, TPAF_FOV_name + "_global_registration_summary.png"), dpi=300)
    plt.show()

#### 方法 2：SIFT + RANSAC

In [ ]:
def rigid_register(tpaf_gray, he_gray):
    
    # Initiate feature extractor
    print("Feature point extraction...")
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(tpaf_gray, None)
    kp2, des2 = sift.detectAndCompute(he_gray, None)
    print(f"Number of keypoints in TPAF: {len(kp1)}, H&E: {len(kp2)}")

    # Match with Lowe's ratio test
    matcher = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), {})
    knn = matcher.knnMatch(des1, des2, k=2)
    good = [m for m,n in knn if m.distance < 0.7 * n.distance]
    print("Descriptor matching completed.")
    print(f"Number of matches: {len(knn)}, Number of good matches: {len(good)}")

    if len(good) < 4:
        raise RuntimeError(f"Too few good matches: {len(good)}")

    src = np.float32([kp1[m.queryIdx].pt for m in good])
    dst = np.float32([kp2[m.trainIdx].pt for m in good])

    # Iterate through all the coordinates of matches and good matches, and plot them in the TPAF image and HE images in two subplots
    print("Plotting matched keypoints...")
    tpaf_pts = np.float32([kp1[m.queryIdx].pt for m in good])
    he_pts = np.float32([kp2[m.trainIdx].pt for m in good])

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    # Plot TPAF image and keypoints
    axes[0].imshow(tpaf_gray, cmap='gray')
    axes[0].set_title('TPAF Keypoints')
    axes[0].scatter(tpaf_pts[:, 0], tpaf_pts[:, 1], c='r', s=10)
    axes[0].axis('off')

    # Plot HE image and keypoints
    axes[1].imshow(he_gray, cmap='gray')
    axes[1].set_title('HE Keypoints')
    axes[1].scatter(he_pts[:, 0], he_pts[:, 1], c='r', s=10)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    model, inliers = cv2.estimateAffine2D(
        src,
        dst,
        method=cv2.USAC_MAGSAC,            # 使用 MSAC（即 M-estimator Sample Consensus）
        ransacReprojThreshold=5.0,       # 残差阈值（像素）
        maxIters=2000,
        confidence=0.99,
        refineIters=10                    # 使用 Levenberg-Marquardt 细化参数
    )

    # OpenCV returns a 2x3 affine matrix, but skimage.transform expects a Transform object.
    # We need to build a skimage.transform.AffineTransform from the matrix.

    if model is None:
        raise RuntimeError("Affine transformation estimation failed.")

    # model is a 2x3 matrix; convert to AffineTransform
    affine_tform = AffineTransform(matrix=np.vstack([model, [0, 0, 1]]))
    tpaf_warped = transform.warp(tpaf_gray, affine_tform.inverse, mode='constant',
                                 cval=0, order=1, preserve_range=True)
    return tpaf_warped.astype(tpaf_gray.dtype), model, inliers


def elastic_register(rigid, he_gray):
    flow = registration.optical_flow_tvl1(he_gray, rigid, attachment=15, tightness=0.3, num_warp=5)
    nr, nc = he_gray.shape
    rr, cc = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')
    r2 = rr + flow[1]
    c2 = cc + flow[0]
    tpaf_elastic = transform.warp(rigid.astype(np.float32), np.stack([r2, c2], axis=-1), order=1, mode='constant', cval=0, preserve_range=True)
    return tpaf_elastic.astype(rigid.dtype)


def crop_border(img, border=50):
    h, w = img.shape
    return img[border:h-border, border:w-border]


def register(tpaf_path, he_path, out_path):
    tpaf_gray = tiff.imread(tpaf_path)
    he_gray = tiff.imread(he_path)

    # improve contrast of TPAF image
    tpaf_gray = cv2.normalize(tpaf_gray, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    tpaf_rigid, model, inliers = rigid_register(tpaf_gray, he_gray)
    #print(f"Rigid transform: rot={np.degrees(model.rotation):.2f}°, trans={model.translation}, inliers={None if inliers is None else inliers.sum()}")
    
    print("Start elastic registration...")
    tpaf_elastic = elastic_register(tpaf_rigid, he_gray)
    registered = crop_border(tpaf_elastic, border=0)

    tiff.imwrite(out_path, registered, dtype=registered.dtype, photometric='minisblack')
    print("Saved registered image with shape:", registered.shape)

    # 可视化结果
    fig, axes = plt.subplots(1, 3, figsize=(15,5))
    axes[0].imshow(tpaf_gray, cmap='gray'); axes[0].set_title('Original TPAF')
    axes[1].imshow(tpaf_rigid, cmap='gray'); axes[1].set_title('After Rigid')
    axes[2].imshow(registered, cmap='gray'); axes[2].set_title('After Elastic')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()
    

In [ ]:
root_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
TPAF_FOV_name = "20240829 750nm 20mW 240817HOC240827-4 Line-02_0001"
og_TPAF_dir = os.path.join(root_dir, "00_og_TPAF")
source_tpaf_dir = os.path.join(root_dir, "01_TPAF_rescaled_gray")
target_he_dir = os.path.join(root_dir, "02_correlation_matching_results/" + TPAF_FOV_name)
output_dir = os.path.join(root_dir, "03_global_registration_results/" + TPAF_FOV_name)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

source_tpaf_name = TPAF_FOV_name + ".tif"
target_he_name = source_tpaf_name[:-4] + "_matched_HE_patch.tif"
og_tpaf_path = os.path.join(og_TPAF_dir, source_tpaf_name)
source_tpaf_path = os.path.join(source_tpaf_dir, source_tpaf_name)
target_he_path = os.path.join(target_he_dir, target_he_name)

corrected_tpaf_path = os.path.join(output_dir, TPAF_FOV_name + "_corrected.tif")

register(source_tpaf_path, target_he_path, corrected_tpaf_path)


#### 方法 3：仅用组织轮廓上的关键点来匹配

In [ ]:
def extract_contour_points(img_gray, canny_thresh=(50,150), max_points=5000):
    edges = cv2.Canny(img_gray, canny_thresh[0], canny_thresh[1])
    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    pts = np.vstack(cnts).squeeze()
    if pts.shape[0] > max_points:
        idx = np.random.choice(pts.shape[0], max_points, replace=False)
        pts = pts[idx]
    return pts  # shape (N,2)

def compute_orb_descriptors(img_gray, pts):
    orb = cv2.ORB_create()
    keypoints = [cv2.KeyPoint(float(x), float(y), 31) for x, y in pts]
    _, descriptors = orb.compute(img_gray, keypoints)
    return keypoints, descriptors

def match_descriptors(des1, des2):
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda m: m.distance)
    return matches

def estimate_rigid(pts_src, pts_dst):
    M, inliers = cv2.estimateAffinePartial2D(pts_src, pts_dst, method=cv2.RANSAC, ransacReprojThreshold=5.0)
    return M, inliers

def rigid_transform_tpafto_he(raw_tpa_gray, target_he_gray):
    pts1 = extract_contour_points(raw_tpa_gray)
    pts2 = extract_contour_points(target_he_gray)
    kp1, des1 = compute_orb_descriptors(raw_tpa_gray, pts1)
    kp2, des2 = compute_orb_descriptors(target_he_gray, pts2)
    matches = match_descriptors(des1, des2)
    src = np.array([kp1[m.queryIdx].pt for m in matches])
    dst = np.array([kp2[m.trainIdx].pt for m in matches])
    
    # iterate through all the coordinates of matches, and plot them in the TPAF image and HE images in two subplots
    print("Plotting matched keypoints...")
    print(f"Number of matches: {len(matches)}")
    print(f"Number of keypoints in TPAF: {len(kp1)}, H&E: {len(kp2)}")
    print(f"Number of points in TPAF contour: {len(pts1)}, H&E contour: {len(pts2)}")
    plot_matches(
        plt.gca(),
        raw_tpa_gray, target_he_gray,
        src, dst,
        np.column_stack((np.arange(len(matches)), np.arange(len(matches)))),
        only_matches=True
    )
    plt.axis('off')
    plt.show()

    M, inliers = estimate_rigid(src, dst)
    return M, matches, inliers, kp1, kp2

def apply_transform_and_show(raw_tpa_color, target_he_color, M):
    rows, cols = target_he_color.shape[:2]
    warped = cv2.warpAffine(raw_tpa_color, M, (cols, rows))
    blend = cv2.addWeighted(warped, 0.5, target_he_color, 0.5, 0)
    return warped, blend

In [ ]:
root_dir = "D:/projects/datasets/virtual_staining/20240817_human-ovarian_primary_site/registration_test"
TPAF_FOV_name = "20240829 750nm 20mW 240817HOC240827-4 Line-02_0001"
og_TPAF_dir = os.path.join(root_dir, "00_og_TPAF")
source_tpaf_dir = os.path.join(root_dir, "01_TPAF_rescaled_gray")
target_he_dir = os.path.join(root_dir, "02_correlation_matching_results/" + TPAF_FOV_name)
output_dir = os.path.join(root_dir, "03_global_registration_results_method3/" + TPAF_FOV_name)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

source_tpaf_name = TPAF_FOV_name + ".tif"
target_he_name = source_tpaf_name[:-4] + "_matched_HE_patch.tif"
og_tpaf_path = os.path.join(og_TPAF_dir, source_tpaf_name)
source_tpaf_path = os.path.join(source_tpaf_dir, source_tpaf_name)
target_he_path = os.path.join(target_he_dir, target_he_name)

corrected_tpaf_path = os.path.join(output_dir, TPAF_FOV_name + "_corrected.tif")
overlay_output_path = os.path.join(output_dir, TPAF_FOV_name + "_overlay.png")


raw_tpaf = tiff.imread(os.path.join(source_tpaf_dir, source_tpaf_name))
target_he = tiff.imread(os.path.join(target_he_dir, target_he_name))
M, matches, inliers, kp1, kp2 = rigid_transform_tpafto_he(raw_tpaf, target_he)
print("Estimated rigid M:", M, "Inliers:", np.sum(inliers))

raw_tpa_color = cv2.cvtColor(raw_tpaf, cv2.COLOR_GRAY2BGR)
target_he_color = cv2.cvtColor(target_he, cv2.COLOR_GRAY2BGR)
warped, blend = apply_transform_and_show(raw_tpa_color, target_he_color, M)
tiff.imwrite(corrected_tpaf_path, warped.astype(np.uint8))
cv2.imwrite("overlay.png", blend)

# visualize the matched keypoints and inliers
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(raw_tpaf, cmap='gray')
ax[0].set_title('TPAF Image')
for m in matches:
    # OpenCV DMatch stores the index of the match in m.queryIdx and m.trainIdx,
    # but inliers is an array of shape (N_matches, 1), so we should use the match index in the matches list.
    # Use enumerate to get the match index.
    for idx, m in enumerate(matches):
        if inliers[idx][0]:
            pt = kp1[m.queryIdx].pt
            ax[0].plot(pt[0], pt[1], 'ro', markersize=2)
        pt = kp1[m.queryIdx].pt
        ax[0].plot(pt[0], pt[1], 'ro', markersize=2)
ax[1].imshow(target_he, cmap='gray')
ax[1].set_title('H&E Image')
for m in matches:
    if inliers[m.queryIdx]:
        pt = kp2[m.trainIdx].pt
        ax[1].plot(pt[0], pt[1], 'ro', markersize=2)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, TPAF_FOV_name + "_matched_keypoints.png"), dpi=300)
plt.show()

## Step 3: Rough Stain Style Transformation

训练初始网络用于颜色映射

使用一个轻量迭代的神经网络（结构如 Fig. 2）进行颜色映射学习，避免学习空间偏移（即保持结构不变，仅做颜色风格转化）。

将荧光图像输入该网络，生成“粗略染色图像”。

## Step 4: Elatic Registration

弹性局部配准（Elastic Registration）

使用弹性图像配准算法对“粗染色图像”与真实明场图像进行局部结构对齐：

分层次（从大到小）地块匹配图像局部区域（块状特征匹配），实现更精细的空间一致性。

将计算得到的**局部变换图（deformation map）**应用于每个明场图像 patch，实现像素级对齐。

## Step 5: Construct Matched TPAF-HE Training Pair for Supervised Virtual Staining Network

构建训练数据对

完成上述步骤后，每对荧光图像和明场组织图像 patch 就准确匹配，可以作为神经网络的输入和标签，用于训练虚拟染色网络。

为生成完整的 WSI，还需对图像进行阴影校正和归一化处理：

每个FOV输入网络前都需：

减去整张图的均值

除以整张图像的标准差 → 实现片内和片间归一化

对每张图像应用阴影校正（修正 FOV 边缘相对强度较低的问题）。